# Audio analysis notes: segmentation, foreground, and cleanup

These notes use the Billie Eilish tracks as pipeline trials. They are not evidence that the same thresholds or musical assumptions will work for Douyin audio. For Douyin, expect short clips, speech, music, effects, edits, and multiple simultaneous sources.


## How should we define a piece or phase?

There are several different meanings of “segment,” and they should not be conflated:

- **Analysis frame/window:** a small, often overlapping interval (for example, 2 seconds with a 1-second hop) used to calculate features. This gives consistent input units for vectorizing and clustering; it does not claim a musical boundary.
- **Acoustic event/scene boundary:** a point where the sound changes, such as speech starting, music changing, a cut, or a loud effect. Voice activity detection (VAD), change-point detection, or source/event classifiers can help locate these.
- **Musical section/phase:** a verse, chorus, intro, or outro. Repetition and musical structure models can suggest these, but chroma alone cannot reliably name them. Human review or a trained section-labeling model is needed for confident labels.

For an initial Douyin workflow, do not require verse/chorus labels before analysis. First identify speech/music/silence and major acoustic changes; then create fixed or speech-aligned windows for comparable feature vectors. Keep timestamps so any result can be traced back to the original audio. If musical phases matter for a later question, run a separate music-structure step on music-dominant spans.

A practical starting point is 2–5 second windows with overlap, adjusted to the task. Short windows localize changes, while overlap reduces the chance that a key phrase or transition is split exactly at a boundary. For ASR and rhetoric analysis, sentence/utterance spans are often better than arbitrary fixed windows. There is no universally correct duration; check several examples and tune it to the content and model.

## Do the existing segment boundaries work?

The current [segment_boundaries.csv](../results/librosa/segment/segment_boundaries.csv) was produced in the clean-analysis notebook from **chroma self-similarity** using `librosa.segment.agglomerative(chroma, k=8)`. In other words, it proposes coarse divisions based on changes in pitch-class patterns, with the number of segments fixed to eight.

The saved boundary times are approximately 0.00, 3.70, 81.28, 102.82, 141.76, 163.87, 184.80, and 189.40 seconds. They are useful as an exploratory view of musical structure, but they are not yet validated phase labels. The widely varying intervals (including a roughly 77-second span) show that these are not suitable as short, comparable clustering windows. The last recorded point is not the audio end: the track summary reports about 205.93 seconds, so a downstream consumer must explicitly include the final tail through the audio duration.

The method does not detect speech, determine foreground/background, or know whether a boundary is a verse/chorus transition. Its fixed `k=8` is a modeling choice, not a discovered fact. Treat the output as candidate structural boundaries, inspect the recurrence plot and listen around candidate points, and compare against other tracks before using it. For Douyin, chroma can be weak or misleading when clips are speech-led, noisy, heavily edited, or contain little tonal music.

So: **yes, it works as a first coarse music-structure experiment; no, it is not enough as the segmentation stage for propaganda/audio-content clustering.** Build a separate timeline for speech activity and acoustic events, then choose windows suited to the speech or other target content.

## How can a machine tell foreground/main content from background or messy audio?

A machine does not inherently know what is important. “Main content” is a task definition, and the system estimates it from measurable cues and models. Also distinguish **foreground** (prominent or close in the mix) from **important content** (relevant to the research question): quiet speech can be important, while loud music may be background.

Useful signals and tools include:

1. **VAD (voice activity detection):** estimates which time intervals contain speech. It helps find speech candidates, but may miss singing, distant/overlapped speech, or distorted speech, and may mark speech-like sounds as speech.
2. **Audio event/source classification and separation:** estimates speech, music, noise, or sources such as vocals and accompaniment. Separation can make a source easier to analyze, but artifacts are possible; separated stems are estimates, not ground truth. A vocals stem can include singing and can leak music/noise.
3. **Acoustic quality measurements:** clipping, silence, signal-to-noise ratio (SNR), loudness, spectral shape, and abrupt changes can flag technical problems or unusual spans. These are quality cues, not reliable judgments of importance. Loudness alone cannot distinguish speech from loud background music.
4. **ASR and transcript checks:** speech recognition can reveal whether a speech candidate contains words and help align utterances. ASR can hallucinate or fail on music, noise, accents, and overlapping speakers, so retain confidence/timestamps and review uncertain samples.
5. **Human review and task labels:** sample clips should be listened to and labeled with a small, explicit scheme (for example speech, music, mixed, silence/noise, uncertain; plus whether speech is relevant). This is how we measure whether automatic “main content” decisions match the research goal.

A robust first pass is to keep the original audio and create timestamped candidate intervals; add VAD and speech/music estimates; optionally separate sources and run ASR; then assign confidence and review uncertain or high-impact spans. Do not automatically discard a span only because it is quiet, noisy, or classified as background: it may contain relevant speech or a meaningful sound cue. Preserve both the original and processed versions, and log processing choices.

## How to clean up messy audio

First decide what “messy” means for the intended analysis. Technical defects (clipping, hum, wind, low volume) differ from a meaningful mixture (speech over music, several speakers, or sound effects). Cleaning should make the target easier to analyze without erasing evidence.

Suggested order:

1. **Inspect and preserve:** keep an unchanged source copy; record duration, sample rate, channels, and provenance. Listen to representative clips and inspect waveform/spectrogram. Do not overwrite the source.
2. **Standardize safely:** convert to the expected sample rate/channel format for each model, check for empty/corrupt files, and avoid clipping during resampling or level adjustment.
3. **Measure and flag:** detect clipped samples, long silence, very low level, abrupt cuts, and likely low-SNR spans. Use flags and confidence rather than silently deleting them.
4. **Separate the task-relevant source:** use VAD for speech timing, speech enhancement for noisy speech, or source separation when speech and music overlap. Compare enhanced/separated results with the original; processing can distort words and transient sounds.
5. **Transcribe and review:** run ASR on speech candidates, align words/utterances to timestamps, and manually inspect uncertain or consequential spans. Keep “unintelligible/uncertain” as a valid result rather than forcing a guess.
6. **Create analysis windows and vectors:** extract features for each timestamped interval from the original and, where useful, from separated stems. Store provenance and quality indicators alongside every vector.

Avoid treating denoising as neutral: a model may suppress quiet voices, alarms, music, or other meaningful signals. For propaganda research, the content and delivery may both matter, so keep separate representations for speech/text, voice/prosody, music, and overall mix rather than cleaning everything into one signal. Compare downstream results with and without enhancement on a reviewed sample.

## Current workflow: implemented trial pipeline and planned next steps

The diagram separates work that already runs in the Billie Eilish trial from work that is still planned. The existing pipeline analyzes music/audio features; it does **not** yet detect propaganda, classify foreground/background/noise, create a unified segment-level vector, or cluster Douyin clips.

```text
IMPLEMENTED TRIAL (Billie Eilish audio)
YouTube/local audio
  -> download/organize audio (yt-dlp in notebook)
  -> Demucs source separation (vocals / no_vocals for bad guy trial)
       -> Basic Pitch on vocals (note events and MIDI)
  -> Librosa analysis of original and separated stems
       -> summary features: RMS, MFCC means, spectral contrast, tempo estimate
       -> chroma profile and key candidates
       -> coarse chroma self-similarity segmentation (fixed k=8)
       -> beat tracking
       -> DTW chroma comparison of original and no_vocals
  -> CSV, plots, stems, and MIDI in results/

PLANNED FOR THE DOUYIN RESEARCH PIPELINE (not implemented yet)
Douyin clips
  -> preserve original + inspect/measure quality
  -> estimate speech/music/events; review uncertain spans
  -> timestamped short windows or speech-aligned utterances
  -> feature/embedding vectors per window (+ text vectors if useful)
  -> scale/normalize and cluster; inspect examples and evaluate
```

The coarse chroma boundaries are candidate musical-structure boundaries only. They are not a substitute for speech/event detection or fixed, comparable analysis windows. The existing DTW output is an alignment/comparison result, not a clustering result.

## Common tools for vectorizing audio (and what each vector represents)

Vectorization means mapping a whole clip or a time window to a fixed-size list of numbers. There is no single universally correct audio vector: the representation determines what “similar” means. For this project, keep timestamps, clip IDs, quality flags, and the source of each vector next to it.

| Approach / common tools | What it produces | Useful for | Important limitation |
|---|---|---|---|
| **Handcrafted acoustic features** — `librosa`, `torchaudio`, Essentia | MFCC, RMS/loudness, spectral centroid/contrast, chroma, pitch, onset/rhythm; summarize each window with mean/std/quantiles | Interpretable acoustic comparisons; a practical baseline and quality checks | Requires feature and window choices; music-oriented features may not represent speech meaning or propaganda rhetoric. MFCC/chroma summaries already exist in the trial, but not yet as a row-per-window clustering matrix. |
| **General audio embeddings** — YAMNet, PANNs, VGGish, OpenL3 | Pretrained neural vectors, often one vector per short frame/window; aggregate frame vectors if a clip-level vector is needed | Broad sound/event or timbral similarity; useful when manual features are insufficient | Models differ in training data, labels, input duration, and intended use. YAMNet is also commonly used for event scores; score outputs are not the same thing as embeddings. Check licensing, model availability, and performance on target clips. |
| **Audio-text embeddings** — CLAP (Contrastive Language-Audio Pretraining) | Audio and text mapped into a shared embedding space | Comparing audio to text descriptions, or exploring broad semantic sound similarity | Similarity to prompts is not a verified content label, and not a reliable standalone propaganda detector. |
| **Speech representations** — wav2vec 2.0, HuBERT, WavLM via Hugging Face/`torchaudio` | Frame-level learned speech representations; pool over an utterance/window for fixed-size vectors | Speech sound/phonetic and voice-delivery patterns | Often requires speech-focused preprocessing and pooling; vectors are not transcript meaning and can encode speaker/channel effects. |
| **Speaker embeddings** — SpeechBrain ECAPA-TDNN, pyannote.audio | A vector intended to represent speaker identity/voice characteristics | Diarization or comparing voice identity across utterances, when appropriate | This is not a topic/content vector; handle identity-related data carefully, and do not infer identity from a cluster without validation. |
| **Speech-to-text then text embeddings** — Whisper/faster-whisper for ASR; Sentence-Transformers for sentence vectors | ASR text with timestamps, then a text vector per utterance or sentence | Topic, claims, repeated slogans, and rhetorical/semantic similarity | ASR errors propagate; text vectors omit vocal delivery, music, and acoustic context. Keep ASR confidence and review uncertain transcripts. |

`librosa` and `torchaudio` are feature/audio-processing libraries, not one-click semantic vectorizers. In the current notebook, `librosa` is already used for handcrafted features; the other model families above are options to evaluate, not tools currently wired into this project.

A sensible baseline is to begin with per-window handcrafted features and one pretrained audio embedding model, then compare results on a small manually reviewed sample. Add ASR plus text embeddings if the research question concerns spoken claims or rhetoric. Keep audio, speech-delivery, speaker, and text representations as separate feature groups initially; compare them individually before fusing, because concatenating unscaled vectors can let one group dominate. Scale numeric features, handle missing values, reduce dimensions only when useful, and use the same preprocessing consistently for every clip. Evaluate clusters by listening to representative and boundary cases, not only by a numeric clustering score.

## 预训练音频 embedding 是什么？以及如何寻找宣传类音频？

### 预训练音频 embedding（预训练音频向量）

预训练模型先在大量音频上学习声音规律。把一段新音频输入模型后，它会输出一串数字，作为这段音频的向量表示（embedding）。可以把它想成模型做出的“数字化声音速写”：向量接近，表示模型认为两段声音在它学到的某些特征上相似；这**不代表**两段音频主题相同，也不证明它们具有某种意图。

对我们的探索用途，embedding 可以帮助把相似片段放在一起，让研究者寻找反复出现的声音或内容模式。不同工具关注的内容不同：音频向量可能反映声学风格；声音事件模型可能提示语音、音乐或环境声；ASR 转写加文本向量可能帮助发现相似说法、主题和叙事。预训练不代表模型理解了我们的研究问题，也不意味着它针对 Douyin 或宣传内容训练过。它提供的是线索，不是结论或标签。开始时不必急着微调模型；先在人工检查过的样本上评估，确认模型有什么用、会犯什么错，再决定是否需要微调。

### 如何寻找和分析宣传类音频？

“宣传”不是单靠音色、音乐或响度就能判定的声学类别。它通常需要结合**说了什么、如何表达、传播的语境和研究定义**来分析。模型可以协助检索、转写、整理相似片段和发现候选模式，但不应仅凭一个音频 embedding 就自动把内容定性为宣传。

1. **先明确操作性定义和范围。** 写清楚你们在这个研究里如何使用“宣传”一词、研究的主题/时间/地区/语言范围，以及哪些内容不纳入。可以把可能观察的修辞线索作为编码项，例如反复口号、情绪诉求、号召行动、群体对立框架、诉诸权威或紧迫感；这些是需要结合语境核验的分析线索，不是自动判定宣传的充分条件。
2. **建立透明、合规、可复查的语料搜集策略。** 从公开且允许研究使用的材料、合规的平台访问方式、新闻/历史档案或研究语料开始。记录 URL 或来源标识、采集日期、标题/描述、语言、时长、账号/频道可公开的上下文，以及纳入/排除原因；遵守平台条款、版权和隐私要求，不尝试绕过访问限制。只用“已知宣传案例”作种子会造成确认偏差；应同时收集相关主题的多种来源和反例/对照材料。
3. **先做小型试点，再扩大搜集。** 选取不同来源、时长、语言和音质的代表性样本，检查能否获得音频、转写和必要的上下文。避免只挑选听起来最激烈或最符合预期的片段。
4. **用模型生成候选，不让模型代替研究判断。** 可对音频做 ASR（中文/目标语言需检查效果），用关键词和语义搜索找候选转写，再用文本向量/聚类发现重复表达和主题；音频事件/音频 embedding 可补充背景音乐、语气、场景或声音风格的相似性。聚类只说明某种表示空间里的相似，不说明其政治立场或宣传意图。
5. **制定标注指南并人工复核。** 对片段记录时间戳、转写、说话/音乐/混合等声音类型、相关修辞线索、证据出处和不确定性；把“无法判断/需上下文”作为有效标签。尽可能让两位标注者独立标注一部分样本，讨论分歧并修订规则；保留反例，并在独立样本上检查一致性。
6. **验证而不是只看聚类图。** 听/读每个簇的代表样本和边界样本，检查是否其实按语言、音质、音乐、频道或重复转载聚在一起。与非宣传或其他类型的相近内容比较，并报告采样局限、误差和不确定性。

因此，一个稳妥的发现流程是：**明确研究定义 → 合规、平衡地收集并记录来源 → ASR/声学分析生成候选 → 向量化与聚类辅助探索 → 人工按指南复核和标注 → 用对照样本验证解释**。聚类可以帮助发现潜在主题和模式；要声称某片段属于宣传，仍需透明的定义、内容/语境证据和可复查的人工判断。

## One-sentence summary

Use coarse structure boundaries as hypotheses, not labels; detect and review speech/music/events first, then make timestamped task-appropriate windows and vectorize them while preserving the original audio and uncertainty about what counts as meaningful foreground.

## Work journal / log

| Timestamp (UTC) | Work completed / current understanding |
|---|---|
| 2026-10-03 00:13–00:22 | Inspected the existing `results/` outputs for the Billie Eilish trial: feature summaries, chroma profiles, Demucs vocal/accompaniment stems, Basic Pitch note events, beat tracking, key candidates, and coarse librosa structural segmentation. Discussed moving toward Douyin audio, short-window features/vectorization/clustering, and the need to distinguish speech/content from background and technical noise. |
| 2026-10-03 00:22 | Reviewed the segmentation code and CSV. Confirmed it uses chroma self-similarity with agglomerative segmentation fixed at `k=8`; boundaries are exploratory musical-structure candidates, not short windows, semantic phases, or speech/background labels. Noted variable span lengths and that the last saved boundary (about 189.40 s) precedes the approximately 205.93 s track end. Added these caveats and a practical foreground/cleanup workflow here. |
| 2026-10-03 00:29 | Added a current-state workflow diagram distinguishing implemented Billie Eilish trial analyses from planned Douyin steps, plus a comparison of common handcrafted, pretrained audio, speech, speaker, and text-vectorization tools. Clarified that no segment-level vector matrix or clustering is implemented yet. |
| 2026-10-03 00:32 | Added a Chinese explanation of pretrained audio embeddings and a discovery workflow for propaganda-related audio, emphasizing transparent definitions, compliant and balanced corpus collection, model-assisted candidate discovery, human review, contextual evidence, and validation against comparison samples. |

Add entries with an ISO 8601 UTC timestamp when meaningful project work is completed. Record what was actually done, the data/model involved, and important caveats; do not treat this journal as a substitute for experiment metadata or reproducible logs.